<a href="https://colab.research.google.com/github/SheffieldML/notebook/blob/main/GPy/sampling_hmc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#A Brief Guide of Using Hybrid Monte Carlo for Gaussian Process Models

### Zhenwen Dai and Mu Niu, November 2014


In [ ]:
try:
    import GPy
except ImportError:
    import subprocess
    import sys
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "GPy[plotting] @ git+https://github.com/SheffieldML/GPy.git@main"]
    )
    import GPy
try:
    import seaborn
except ImportError:
    import subprocess
    import sys
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", 'seaborn']
    )
    import seaborn
try:
    import pods
except ImportError:
    import subprocess
    import sys
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "pods"]
    )
    import pods



In [ ]:
from matplotlib import pyplot as plt
import numpy as np
import GPy
#%config InlineBackend.figure_format = 'svg'


## Example 1  HMC Inference for GP Regression

Let's first generate some synthetic data.

In [ ]:
# Let's make some synthetic data
x = np.linspace(0.,2*np.pi,100)[:,None]
y = -np.cos(x)+np.random.randn(*x.shape)*0.3+1
_ = plt.plot(x,y,'.')


Let us Make a GP Regression model and give some general prior distributions to model parameters.

In [ ]:
# Make a GP regression model
m = GPy.models.GPRegression(x,y)
# Give some general prior distributions for model parameters
m.kern.lengthscale.set_prior(GPy.priors.Gamma.from_EV(1.,10.))
m.kern.variance.set_prior(GPy.priors.Gamma.from_EV(1.,10.))
m.likelihood.variance.set_prior(GPy.priors.Gamma.from_EV(1.,10.))
_=m.plot()

###Sampling the posterior distribution of model parameters

Draw 1000 samples from the GP model

In [ ]:
hmc = GPy.inference.mcmc.HMC(m,stepsize=5e-2)
s = hmc.sample(num_samples=1000) # Burnin
s = hmc.sample(num_samples=1000)

Plot the samples:

In [ ]:
plt.plot(s)


Plot the posterior marginal distribution of model parameters:

In [ ]:
labels = ['kern variance', 'kern lengthscale','noise variance']
samples = s[300:] # cut out the burn-in period
from scipy import stats
xmin = samples.min()
xmax = samples.max()
xs = np.linspace(xmin,xmax,100)
for i in range(samples.shape[1]):
    kernel = stats.gaussian_kde(samples[:,i])
    plt.plot(xs,kernel(xs),label=labels[i])
_ = plt.legend()


Plot the model parameters (lengthscale, variance and noise variance) against each other:

In [ ]:
fig = plt.figure(figsize=(14,4))
ax = fig.add_subplot(131)
_=ax.plot(samples[:,0],samples[:,1],'.')
ax.set_xlabel(labels[0]); ax.set_ylabel(labels[1])
ax = fig.add_subplot(132)
_=ax.plot(samples[:,1],samples[:,2],'.')
ax.set_xlabel(labels[1]); ax.set_ylabel(labels[2])
ax = fig.add_subplot(133)
_=ax.plot(samples[:,0],samples[:,2],'.')
ax.set_xlabel(labels[0]); ax.set_ylabel(labels[2])


By setting the model parameters to the posterior mean, we can visualize the model fit:

In [ ]:
# Set the model parameters as the posterior mean
m.kern.variance[:] = samples[:,0].mean()
m.kern.lengthscale[:] = samples[:,1].mean()
m.likelihood.variance[:] = samples[:,2].mean()
print(m)
_=m.plot()

### Sample the posterior distribution of X given some new Y

Given some new observations, inferring the posterior distribution of the corresponding inputs is difficult, because it can lead to multi-modal distributions.

Assume we have a new observation $1.5$, and try to infer its input distribution.

In [ ]:
y_new = np.array([1.5])[:,None]

Generate the inference model for the new observations. The GP supplies $p(Y\mid X)$ through the covariance evaluated at the unknown inputs $X$; by default `infer_newX` has a flat prior on $X$, so the red-dot MAP is a mode of the likelihood only. That MAP often sits on just one of several modes.

In [ ]:
x_new,mi = m.infer_newX(y_new)
print(mi)
m.plot()
plt.plot(x_new,y_new,'or')


For HMC we need a proper prior on the inputs $X$ as well (the covariance Gaussian is on $f\mid X$, not on $X$). Place a Gaussian prior with mean and standard deviation matching the observed training inputs, then draw 10,000 samples from $p(X\mid Y)\propto p(Y\mid X)\,p(X)$:

In [ ]:
# Covariance terms already encode p(Y|X); set p(X) so the sampler has a proper posterior.
x_obs = np.asarray(m.X, dtype=float).ravel()
mi.X.set_prior(GPy.priors.Gaussian(float(x_obs.mean()), float(x_obs.std())))

hmc_new = GPy.inference.mcmc.HMC(mi, stepsize=2e-1)
s_new = hmc_new.sample(num_samples=10000, hmc_iters=10)

Plot the samples:

In [ ]:
_ = plt.plot(s_new[:,:])


Plot the marginal distribution of inferred inputs. The two modes of inputs are clearly visible from the sampled posterior distribution.

In [ ]:
from scipy import stats
samples_new = s_new[:]
xmin = samples_new.min()
xmax = samples_new.max()
xs = np.linspace(xmin,xmax,100)
for i in range(samples_new.shape[1]):
    kernel = stats.gaussian_kde(samples_new[:,i])
    plt.plot(xs,kernel(xs))


# Example 2 HMC for lengthscale and variance with marathon data

`GPy.examples.regression.olympic_marathon_men` loads the series via [`pods`](https://github.com/lawrennd/ods) (`pods.datasets.olympic_marathon_men`). The bootstrap cell installs `pods` when it is missing.

##### we set prior for lengthscale and variance of kernel. The mean of the prior is close to the result of GP optimisation. we then allow a big variance. In the case below,we set gamma prior to lengthscale and variance. E(lengthscale) = 120, Var(lengthscale)=2000, E(variance) = 25, Var(variance) = 150

In [ ]:
m = GPy.examples.regression.olympic_marathon_men(plot=False)
# Priors on every unfixed parameter (including noise) so HMC includes transform Jacobians.
# warning=False: already +ve-constrained; Gamma.set_prior would otherwise log a reconstraining WARNING.
m.kern.variance.set_prior(GPy.priors.Gamma.from_EV(25., 150.), warning=False)
m.kern.lengthscale.set_prior(GPy.priors.Gamma.from_EV(120., 2000.), warning=False)
m.Gaussian_noise.variance.set_prior(GPy.priors.Gamma.from_EV(1., 10.), warning=False)
print(m)

### Run HMC

##### we plot the full length of hmc iteration. The first 5000 could be burn in stage and can be ignored.

In [ ]:
# initialise hmc
hmc = GPy.inference.mcmc.HMC(m,stepsize=2e-1)
# run hmc
t = hmc.sample(num_samples=20000,hmc_iters=20)

# Sample parameters
#hmc = GPy.inference.optimization.HMC(m, stepsize=5e-1)
#t = hmc.sample(m_iters=50000,hmc_iters=20)

_=plt.plot(t)


In [ ]:
print(t.mean(axis=0))
print(t.std(axis=0))
_=plt.hist(t[:,:2],50)


## Seaborn
Using Seaborn for plotting distributions over Hyperparameters:

In [ ]:
import seaborn as sns, pandas as pd
plt.rcParams['text.usetex'] = False

In [ ]:
df = pd.DataFrame(t, columns=m.parameter_names_flat())
var_col = next(c for c in df.columns if c.endswith('rbf.variance'))
ls_col = next(c for c in df.columns if c.endswith('rbf.lengthscale'))
ax = sns.kdeplot(df[var_col], color="b", fill=True)
ax = sns.kdeplot(df[ls_col], color="r", fill=True)

In [ ]:
sns.set(style="white", color_codes=True)
_ = sns.jointplot(data=df, x=var_col, y=ls_col, kind="hex")

In [ ]:
# Posterior predictive from HMC hyperparameter samples
burn = 5000
thin = 20  # every 20th sample; raise if this is slow
samples = t[burn::thin]

Xstar = np.linspace(1850, 2050, 200)[:, None]
means = np.empty((len(samples), Xstar.shape[0]))
vars_ = np.empty_like(means)

for i, theta in enumerate(samples):
    m.kern.variance[:] = theta[0]
    m.kern.lengthscale[:] = theta[1]
    m.Gaussian_noise.variance[:] = theta[2]
    mu, var = m.predict(Xstar)
    means[i] = mu[:, 0]
    vars_[i] = var[:, 0]

# Law of total variance: E[var|θ] + Var[mean|θ]
mu_bar = means.mean(axis=0)
var_bar = vars_.mean(axis=0) + means.var(axis=0)
std_bar = np.sqrt(var_bar)

plt.figure(figsize=(8, 4))
plt.plot(m.X, m.Y, "kx", label="data")
plt.plot(Xstar[:, 0], mu_bar, "b-", label="posterior predictive mean")
plt.fill_between(
    Xstar[:, 0],
    mu_bar - 2 * std_bar,
    mu_bar + 2 * std_bar,
    color="b",
    alpha=0.2,
    label=r"$\pm 2\sigma$ (incl. hyperparam. uncertainty)",
)
plt.xlabel("year")
plt.legend()
plt.show()